In [ ]:
# For users following this tutorial:
# !pip install ragpill langchain-huggingface sentence-transformers ipywidgets
# Or if you're developing locally with this repo:
# !uv sync --group docs

## Settings

Context: The settings are optional but an easy way to configure for standard use cases.
If you need more control over the client that is passed to [LLMJudge](../api/evaluators.md#llmjudge), you can also pass a pydantic-ai model directly.

To set up mlflow tracking and LLMJudge, you can create a `.env` file in the root of your project with the following content:

```env
# --- MLflow ---
# URI of your MLflow tracking server (default: http://localhost:5000)
MLFLOW_RAGPILL_TRACKING_URI=http://localhost:5000

# Name of the MLflow experiment (default: ragpill_experiment)
MLFLOW_RAGPILL_EXPERIMENT_NAME=my_experiment

# Human-readable description attached to each MLflow run (default: RAGPill Evaluation Run)
MLFLOW_RAGPILL_RUN_DESCRIPTION=My Evaluation Run

# Optional: credentials for a secured MLflow server
MLFLOW_TRACKING_USERNAME=your_username
MLFLOW_TRACKING_PASSWORD=your_password

# --- LLMJudge ---
# Model name passed to the LLM provider (default: gpt-4o)
RAGPILL_LLMJUDGE_MODEL_NAME=gpt-4o

# Sampling temperature for the judge model (default: 0.0)
RAGPILL_LLMJUDGE_TEMPERATURE=0.0

# Optional: base URL for a custom or self-hosted LLM API endpoint.
# Falls back to OPENAI_BASE_URL, then https://api.openai.com/v1
RAGPILL_LLMJUDGE_BASE_URL=https://your-llm-endpoint/v1

# Optional: API key for the LLM service.
# Falls back to OPENAI_API_KEY if not set.
RAGPILL_LLMJUDGE_API_KEY=sk-...
```

The cell below loads these settings and prints the resolved values so you can verify your configuration before running evaluations.

In [ ]:
from ragpill.settings import MLFlowSettings, LLMJudgeSettings
from pprint import pprint
mlflow_settings = MLFlowSettings()
pprint(mlflow_settings)
llm_judge_settings = LLMJudgeSettings()
pprint(llm_judge_settings)

## Defining the Testset



For more details on best practices for creating a testset, see the [Testsets](../guide/testsets.md ) guide.

Here we showcase the use of tags, attributes and "expected" attribute and how its propagated as default from Case to Evaluator.

In [ ]:
from pydantic_evals import Dataset, Case
from ragpill import LLMJudge, TestCaseMetadata, RegexInOutputEvaluator
testset = Dataset(
    name="testset1",
    cases=[
        Case(
            inputs="What is the capital of France?",
            metadata=TestCaseMetadata(tags=["geography"], attributes={"author": "janine"}),
            evaluators=[
                LLMJudge(rubric="Paris is the capital of France"),
            ]
        ),
        Case(
            inputs="How many r`s are in 'strawberry'? State only the number",
            metadata=TestCaseMetadata(expected=False, attributes={"author": "joel"}),
            evaluators=[
                RegexInOutputEvaluator(pattern="1"),
                RegexInOutputEvaluator(pattern="2"),
                RegexInOutputEvaluator(pattern="3", expected=True),
                RegexInOutputEvaluator(pattern="4"),
            ]
        )
    ]
)

## Creating the Task

In the end we just need a function (or if async: a coroutine) that takes in an inputs-`str` (usually a question from the user) and returns a `str` (usually the answer).
We will wrap a very simple [pydantic-ai Agent](https://ai.pydantic.dev/agent/)

In [ ]:
from openai import AsyncOpenAI
import os
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

api_key = os.environ["RAGPILL_LLMJUDGE_API_KEY"]
base_url = os.environ["RAGPILL_LLMJUDGE_BASE_URL"]
model_name = os.environ["RAGPILL_LLMJUDGE_MODEL_NAME"]
system_prompt = "You are a helpful assistant"
temperature = 0.2

openai_client = AsyncOpenAI(
    max_retries=3,
    base_url=base_url,
    api_key=api_key,
)
pyai_llm_model = OpenAIChatModel(
    model_name, provider=OpenAIProvider(openai_client=openai_client), settings={"temperature": temperature}
)

agent = Agent(model=pyai_llm_model, system_prompt=system_prompt)

## Running the Evaluation

In [ ]:
import mlflow
from mlflow import MlflowClient

tracking_uri = mlflow_settings.ragpill_tracking_uri
try:
    client = MlflowClient(tracking_uri=tracking_uri)
    client.search_experiments()
    print(f"MLflow server reachable at {tracking_uri}")
except Exception:
    print(
        f"⚠️  MLflow server not reachable at {tracking_uri}.\n"
        "To track task runs start the server first:\n\n"
        "    mlflow server --backend-store-uri sqlite:///mlflow.db\n\n"
        "Then re-run this cell before continuing."
    )
    raise

In [ ]:
from IPython.display import display

async def task(question: str) -> str:
    result = await agent.run(question)
    return result.output

result = await task("Whats the capital of France?")
display(result)

In [ ]:
from ragpill import evaluate_testset_with_mlflow

async def task(question: str) -> str:
    result = await agent.run(question)
    return result.output


results_df = await evaluate_testset_with_mlflow(
    testset=testset,
    task=task,
    model_params={
        "system_prompt": system_prompt,
        "temperature": str(temperature),
        "model_name": model_name,
    }
)
results_df